# Análisis de sentimientos con Python

Este notebook replica la lógica pedagógica de la herramienta práctica de la unidad 6:

1. crear un conjunto pequeño de reseñas;
2. tokenizar el texto;
3. asignar un puntaje de sentimiento base;
4. corregir parcialmente negaciones;
5. mirar palabras frecuentes;
6. y luego pasar a un conjunto más grande de reseñas reales.


In [ ]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd


## Paso 1. Crear un conjunto pequeño de reseñas

Partimos con cinco reseñas ficticias en inglés para que el flujo se vea con claridad.


In [ ]:
reviews = pd.DataFrame({
    "customer": ["Customer 1", "Customer 2", "Customer 3", "Customer 4", "Customer 5"],
    "review": [
        "Excellent product, I highly recommend it!",
        "The product was good but the delivery was slow",
        "I am not satisfied with the product, I expected better quality",
        "The product arrived in perfect condition and in record time",
        "I would not recommend this product to others, it did not meet my expectations"
    ]
})

reviews


## Paso 2. Tokenizar el texto

Tokenizar significa separar el texto en palabras para poder trabajar luego con diccionarios o reglas.


In [ ]:
token_pattern = re.compile(r"[a-z']+")

def tokenize(text):
    return token_pattern.findall(text.lower())

tokens = (
    reviews.assign(word=reviews["review"].apply(tokenize))
    .explode("word")
    [["customer", "word"]]
)

tokens.head(10)


## Paso 3. Asignar un puntaje de sentimiento base

Usamos un diccionario pequeño para mostrar la idea de un enfoque léxico. No intenta ser perfecto; justamente sirve para ver sus límites.


In [ ]:
sentiment_lexicon = {
    "excellent": 1,
    "recommend": 1,
    "good": 1,
    "better": 1,
    "perfect": 1,
    "slow": -1,
    "bad": -1,
}

scored_tokens = tokens.assign(value=tokens["word"].map(sentiment_lexicon).fillna(0))
reviews_scored = (
    scored_tokens.groupby("customer", as_index=False)["value"]
    .sum()
    .rename(columns={"value": "sentiment_score"})
)

reviews_scored


### ¿Qué problema aparece aquí?

Las reseñas con expresiones como `not satisfied` o `not recommend` siguen viéndose demasiado positivas. Eso pasa porque el método está leyendo palabras aisladas, no la frase completa.


In [ ]:
negation_examples = [
    phrase for phrase in reviews["review"]
    if "not" in phrase.lower() or "did not" in phrase.lower()
]

negation_examples


## Paso 4. Corregir parcialmente negaciones

Una solución simple consiste en reemplazar algunas expresiones problemáticas por términos negativos más directos.


In [ ]:
replacements = {
    "not satisfied": "bad",
    "not recommend": "bad",
    "did not meet my expectations": "bad",
    "expected better quality": "bad"
}

reviews_fixed = reviews.copy()
for old, new in replacements.items():
    reviews_fixed["review_fixed"] = reviews_fixed.get("review_fixed", reviews_fixed["review"])
    reviews_fixed["review_fixed"] = reviews_fixed["review_fixed"].str.replace(old, new, regex=False)

reviews_fixed[["customer", "review", "review_fixed"]]


In [ ]:
tokens_fixed = (
    reviews_fixed.assign(word=reviews_fixed["review_fixed"].apply(tokenize))
    .explode("word")
    [["customer", "word"]]
)

scored_tokens_fixed = tokens_fixed.assign(value=tokens_fixed["word"].map(sentiment_lexicon).fillna(0))
reviews_scored_fixed = (
    scored_tokens_fixed.groupby("customer", as_index=False)["value"]
    .sum()
    .rename(columns={"value": "sentiment_score"})
)

reviews_scored_fixed


## Paso 5. Mirar palabras frecuentes

Además del puntaje, muchas veces conviene mirar qué palabras aparecen con más frecuencia.


In [ ]:
stop_words = {
    "the", "i", "it", "to", "with", "am", "was", "but", "this", "a", "in", "and", "my", "are"
}
ignore_words = {"product"}

freq = Counter(
    word
    for word in tokens["word"]
    if word not in stop_words and word not in ignore_words
)

freq_table = pd.DataFrame(freq.most_common(), columns=["word", "n"])
freq_table.head(10)


## Paso 6. Pasar a un conjunto más grande de reseñas reales

El notebook original también incluía un segundo ejemplo con reseñas de audífonos. Aquí reutilizamos el mismo dataset para mostrar cómo cambia el trabajo cuando ya no son solo cinco textos.


In [ ]:
local_path = Path('data_bose.csv')
remote_url = 'https://raw.githubusercontent.com/melanieoyarzun/herramientas_aplicadas/refs/heads/main/data/data_bose.csv'

if local_path.exists():
    bose = pd.read_csv(local_path)
else:
    bose = pd.read_csv(remote_url)

bose = bose[bose['Verified_Purchase'] == True].copy()

bose[["reviewRating", "reviewHeadline", "reviewFormat"]].head(3)


In [ ]:
bose_text = (
    bose[["reviewText"]]
    .dropna()
    .rename(columns={"reviewText": "review"})
    .head(500)
)

bose_tokens = (
    bose_text.assign(word=bose_text["review"].astype(str).apply(tokenize))
    .explode("word")
    [["word"]]
)

bose_freq = Counter(word for word in bose_tokens["word"] if word not in stop_words)
pd.DataFrame(bose_freq.most_common(15), columns=["word", "n"])


## Cierre

Este ejemplo deja ver una idea importante de la unidad:

- un análisis de texto puede comenzar con reglas y diccionarios simples;
- pero rápidamente aparecen fenómenos lingüísticos que exigen más contexto;
- y por eso hoy conviven enfoques clásicos con modelos más potentes como embeddings y LLMs.
